<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [4]:
# Installation silencieuse des outils indispensables
%pip install -q transformers torch

import torch
import transformers

In [6]:
from transformers import AutoTokenizer

# 1. Chargement du tokeniseur officiel de BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Remplacement du TODO par ta phrase personnalisée
sample_sentence = "The hospital ordered a new ventilator."
print(f"Phrase originale : '{sample_sentence}'\n")

# 3. Découpage brut en tokens textuels (Algorithme WordPiece)
tokens = tokenizer.tokenize(sample_sentence)
print("1. Liste des tokens textuels générés par BERT :")
print(tokens)

# 4. Conversion complète en IDs numériques (avec les tokens spéciaux [CLS] et [SEP])
encoded_input = tokenizer(sample_sentence)
print("\n2. Dictionnaire final généré pour l'entrée du modèle :")
print(encoded_input)

Phrase originale : 'The hospital ordered a new ventilator.'

1. Liste des tokens textuels générés par BERT :
['the', 'hospital', 'ordered', 'a', 'new', 'vent', '##ila', '##tor', '.']

2. Dictionnaire final généré pour l'entrée du modèle :
{'input_ids': [101, 1996, 2902, 3641, 1037, 2047, 18834, 11733, 4263, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# La phrase exemple définie à l'étape précédente
sample_sentence = "The hospital ordered a new ventilator."

# Encodage complet avec gestion automatique du padding et de la troncature
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,      # Ajoute [CLS] et [SEP]
    padding="max_length",         # Active le padding dynamique
    truncation=True,              # Coupe si la phrase dépasse max_length
    max_length=24,                # TODO: Ajusté à 24 pour observer les jetons de remplissage
    return_attention_mask=True,   # Génère le masque d'attention binaire
    return_tensors="pt"           # Format de sortie : Tenseurs PyTorch
)

# Extraction des structures de données pour l'affichage en liste standard
input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Alignement vertical propre sous forme de table
print(f"{'index':>5} | {'token':<12} | {'id':>5}")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

# Affichage des métriques de contrôle demandées par l'exercice
print("\n" + "="*45)
print("Attention mask:", encoding["attention_mask"][0].tolist())

special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        |    id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | hospital     |  2902
    3 | ordered      |  3641
    4 | a            |  1037
    5 | new          |  2047
    6 | vent         | 18834
    7 | ##ila        | 11733
    8 | ##tor        |  4263
    9 | .            |  1012
   10 | [SEP]        |   102
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (10, '[SEP]'), (11, '[PAD]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[P

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.
1. Behavior of [CLS] and [SEP] Inside the EncoderInside BERT's bidirectional encoder, special tokens are not merely static boundary markers; they participate dynamically in the self-attention mechanism:[CLS] (Classification): Positioned at index 0, this token is engineered to act as a global semantic aggregator. Because BERT uses non-directional (bidirectional) self-attention, the embedding for [CLS] updates by looking at every single token in the sequence simultaneously across all encoder layers. By the time it reaches the final layer, the vector at the [CLS] position has absorbed a compressed, holistic representation of the entire sentence's meaning. This makes it the ideal anchor to plug directly into a downstream classification head (like sentiment analysis).[SEP] (Separator): This token serves as a structural boundary force. In single-sentence tasks, it signals the exact termination of the textual payload. In sentence-pair tasks (like Question Answering), it breaks the sequence into distinct segments. During the self-attention pass, it allows tokens to mathematically distinguish whether a neighboring word belongs to Sentence $A$ or Sentence $B$, preventing contextual bleed between separate statements.
 2. How the Attention Mask Hides Padded PositionsThe self-attention mechanism calculates how much focus one token should place on another by computing raw dot-product similarity scores ($QK^T$). Left unmanaged, the model would waste computational capacity calculating semantic relationships between real words and meaningless [PAD] tokens.The Attention Mask acts as a mathematical shield right before the Softmax normalization step:Raw Score Generation: The model computes raw alignment scores for all tokens, including padding slots.Mask Injection: For any position where the attention mask is 0, the system overrides the raw score by adding a massive negative value (typically $-10000.0$).Softmax Suppression: The scaled scores are passed through the Softmax function:$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$$Because $e^{-10000.0}$ approaches exactly $0$, the resulting attention weight assigned to that padded position drops to $0.0$.Consequently, during the final step where the values ($V$) are mixed, the padded positions are multiplied by zero. They exert absolutely no influence over the contextual updates of the real vocabulary, rendering them mathematically invisible.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [8]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "TODO: add a sentence whose sentiment you want to test"
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9761614203453064}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?
1. Label Alignment with ExpectationAnswer: Yes, the predicted POSITIVE label matches my expectation perfectly.Why: The test sentence contains clear, unambiguous positive indicators like "highly efficient" and "drastically reduces patient wait times". Because DistilBERT uses bidirectional self-attention, it doesn't just pick up on these keywords in isolation; it processes the full context of the statement. It successfully connects these positive attributes directly to the subject ("the emergency room layout"), correctly identifying the overall appreciative tone of the feedback.
 2. Model Confidence and Score InterpretationModel Confidence: The model is exceptionally confident, returning a score of $99.98\%$ (or $0.9998$).What the Score Tells Us: In a binary classification setup (Positive vs. Negative), the score represents the normalized probability distribution calculated by the final Softmax layer over the target classes:$$\text{Softmax}(\text{logits}) = [P(\text{Negative}), P(\text{Positive})]$$A score of $99.98\%$ for the positive class indicates that the raw numerical value (logit) calculated for positive sentiment was massively higher than the negative logit. This tells us that the structural patterns in our sentence closely match the clear-cut examples of positive text that the model memorized during its fine-tuning phase, leaving virtually zero mathematical ambiguity.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [12]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict, Any

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        clean_text = text.strip()
        return self.tokenizer(
            clean_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def predict(self, text: str) -> Dict[str, Any]:
        inputs = self.preprocess(text)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1)[0]
        predicted_idx = torch.argmax(probabilities).item()
        label = self.model.config.id2label[predicted_idx]
        confidence_score = probabilities[predicted_idx].item()
        return {"label": label, "confidence": confidence_score}

In [13]:
# Instanciation de l'analyseur personnalisé (la classe construite précédemment)
analyzer = BERTSentimentAnalyzer()

# Définition des phrases de test requises par le TODO
samples = [
    "The customer service was absolutely fantastic and the shipping was incredibly fast!",
    "The product arrived completely broken and the software keeps crashing constantly."
]

print("=== Phase de Test de l'Analyseur Personnalisé ===")
for text in samples:
    print(f"\n Texte analysé : \"{text}\"")

    # Appel de la méthode de prédiction que nous avons codée
    result = analyzer.predict(text)

    # Affichage propre des résultats
    print(f" Sentiment Détecté : {result['label']}")
    print(f" Score de Certitude : {result['confidence']:.2%}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== Phase de Test de l'Analyseur Personnalisé ===

 Texte analysé : "The customer service was absolutely fantastic and the shipping was incredibly fast!"
 Sentiment Détecté : POSITIVE
 Score de Certitude : 99.98%

 Texte analysé : "The product arrived completely broken and the software keeps crashing constantly."
 Sentiment Détecté : NEGATIVE
 Score de Certitude : 99.95%


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [16]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from typing import List, Dict, Any

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # 1. Détection du matériel (GPU si disponible, sinon CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 2. Chargement du tokeniseur et du modèle de classification de tokens
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # 3. Envoi du modèle sur le matériel et bascule en mode évaluation
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str) -> List[Dict[str, Any]]:
        # Encodage du texte en récupérant l'offset_mapping (coordonnées de chaque token)
        inputs = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        offset_mapping = inputs["offset_mapping"][0].tolist()
        input_ids = inputs["input_ids"][0].tolist()

        # Préparation des entrées pour le modèle (sans l'offset_mapping)
        model_inputs = {k: v.to(self.device) for k, v in inputs.items() if k != "offset_mapping"}

        # Passe avant sans calcul de gradients
        with torch.no_grad():
            outputs = self.model(**model_inputs)

        # Extraction de l'index de classe gagnant pour chaque token
        predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
        labels = [self.model.config.id2label[p] for p in predictions]
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids)

        entities = []
        current_entity = None

        # Parcours des tokens pour fusionner les WordPieces et reconstruire les entités
        for i in range(len(tokens)):
            token = tokens[i]
            label = labels[i]
            start_char, end_char = offset_mapping[i]

            # Ignorer les jetons spéciaux de structure ([CLS], [SEP])
            if start_char == 0 and end_char == 0:
                continue

            # Gestion des sous-mots (WordPieces) commençant par "##"
            if token.startswith("##") and current_entity is not None:
                current_entity["end"] = end_char
                continue

            # Si le token appartient à une entité (ex: B-PER, I-PER, etc.)
            if label != "O":
                clean_label = label.split("-")[-1]

                # Si on est déjà en train de suivre une entité de la même catégorie
                if current_entity and current_entity["entity"] == clean_label:
                    current_entity["end"] = end_char
                else:
                    if current_entity:
                        entities.append(current_entity)

                    current_entity = {
                        "text": "",  # Extrait proprement depuis le texte original à la fin
                        "entity": clean_label,
                        "start": start_char,
                        "end": end_char
                    }
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        # Extraction propre du texte original en utilisant les coordonnées
        for ent in entities:
            ent["text"] = text[ent["start"]:ent["end"]]

        return entities

In [17]:
# 1. Instanciation de l'analyseur NER personnalisé
ner = BERTNamedEntityRecognizer()

# 2. Remplacement du TODO par un paragraphe riche en entités nommées
sample_text = "Tomorrow, Dr. Robert Chen will fly from Paris to New York to attend an artificial intelligence summit organized by Google."

print("=== Phase de Test du Système NER Personnalisé ===")
print(f"Texte analysé : \"{sample_text}\"\n")

# 3. Exécution de la reconnaissance d'entités
detected_entities = ner.recognize(sample_text)

# 4. Affichage propre et structuré des résultats sous forme de dictionnaire
import json
print(json.dumps(detected_entities, indent=2))

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Phase de Test du Système NER Personnalisé ===
Texte analysé : "Tomorrow, Dr. Robert Chen will fly from Paris to New York to attend an artificial intelligence summit organized by Google."

[
  {
    "text": "Robert Chen",
    "entity": "PER",
    "start": 14,
    "end": 25
  },
  {
    "text": "Paris",
    "entity": "LOC",
    "start": 40,
    "end": 45
  },
  {
    "text": "New York",
    "entity": "LOC",
    "start": 49,
    "end": 57
  },
  {
    "text": "Google",
    "entity": "ORG",
    "start": 115,
    "end": 121
  }
]


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |
CategoryBERTGPTArchitectureEncoder-only framework leveraging bidirectional context.Decoder-only framework leveraging causal (autoregressive) left-to-right context.Primary purposeDeep semantic extraction and contextual understanding of a full sequence.Generative sequence completion and predictive next-token text synthesis.Typical use casesSentiment Analysis, Named Entity Recognition (NER), Question Answering.Open-ended Text Generation, Conversational Agents (Chatbots), Code Synthesis.StrengthsCaptures non-directional, simultaneous relations on both sides of a token.Exceptional fluency, zero-shot/few-shot task adaptation, and creative generation.WeaknessesStructurally incapable of text generation tasks due to look-ahead visibility.Prone to hallucinations and highly susceptible to directional drift over long loops.

 Core Architectural InsightBERT (Bidirectional Encoder Representations from Transformers): Evaluates a sentence by calculating attention weights in both directions simultaneously ($T_{-n} \leftrightarrow T \leftrightarrow T_{+n}$). This global visibility makes it highly precise at extracting nuanced relationships within a static block of text, but leaves it mathematically incapable of fluidly generating new tokens.GPT (Generative Pre-trained Transformer): Employs masked self-attention, meaning it applies a strict diagonal constraint mask that prevents the model from seeing future tokens ($T_{-n} \rightarrow T$). It predicts the future by analyzing the absolute historical sequence, making it the industry standard for generative language tasks.

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.
1. Encoding Queries and Documents with BERTBERT transforms raw text chunks and user search queries into dense, high-dimensional vector representations (embeddings). During document ingestion, long text sources are parsed into smaller paragraphs, and BERT processes each chunk bi-directionally to generate a fixed-size mathematical representation—often pulling the final hidden state of the special [CLS] token or using mean pooling across all token embeddings. When a user submits a search query, it passes through the exact same BERT encoder model. Because BERT excels at mapping semantic nuance rather than just matching surface keywords, it ensures that a query like "ventilator failure protocols" and a document chunk reading "emergency procedures for respiratory device breakdowns" are mapped to nearly identical coordinates in the vector space.
2. Storing and Searching in a Vector DatabaseOnce BERT converts the text chunks into dense arrays of floating-point numbers (typically 768 or 1024 dimensions), these vector embeddings are stored in a specialized vector database (such as Pinecone, Milvus, or Qdrant) alongside their raw text metadata. When a user query vector arrives, the database does not perform a traditional SQL keyword search. Instead, it runs mathematical similarity algorithms—most commonly Cosine Similarity or Dot Product calculations—to determine the distance between the query vector and the stored document vectors. Using efficient index frameworks like Hierarchical Navigable Small World (HNSW), the database instantly surfaces the top $K$ document chunks whose semantic vectors are physically closest to the query vector.
3. Handing Context to the Generative Model (GPT)After the vector database identifies the top $K$ most relevant text passages, these raw text strings are extracted from the metadata and injected into a structured prompt template alongside the user's original query. This combined payload forms an augmented context window. The final prompt is formatted cleanly (e.g., "Answer the query using only the following context: [Retrieved Passages] ... User Query: [Query]") and dispatched via an API call to a generative decoder model like GPT. GPT reads this injected ground-truth context, shifts into an in-context learning state, and uses its autoregressive generation capabilities to synthesize a highly accurate, fluent answer completely free of hallucinations.
4. Concrete Application Example: Medical Regulatory ComplianceA premier application where a BERT-driven RAG system is vital is a Medical Device Audit and Compliance Copilot. Medical manufacturers must navigate thousands of pages of deeply complex, highly technical regulatory paperwork (such as FDA guidelines and ISO standards).The Retrieval Layer: A bi-directionally trained Bi-Encoder (like SBERT) excels at capturing dense, complex medical semantics, ensuring that when an auditor asks about "biocompatibility testing thresholds for implants," the system retrieves exact engineering sub-clauses even if the wording varies.The Generation Layer: GPT takes those exact technical standard clauses and drafts a flawless, audit-ready compliance report. This hybrid pipeline ensures the system inherits BERT's precise, domain-specific retrieval capabilities along with GPT's superior synthesis, protecting healthcare organizations from costly compliance penalties.